# Baselines: bill advancement and cosponsor prediction

Trains and compares every baseline on the leak-free Congress graph. All the logic lives in the `baselines`
package next to this notebook; each section below is a few lines of explanation and a few calls.
Full rationale, citations and results: `docs/BASELINES.md`.

- **Bill advancement:** is a law-capable bill reported out of committee within 180 days? Metric: AUROC (and AP).
- **Cosponsor prediction:** rank the eligible legislators by who cosponsors a new bill in the next 30 days.
  Metric: Recall@20 over the full eligible pool (and Recall@10, NDCG@20).
- **Split:** by Congress. Simple mode = fold 117: train up to 115, validate on 116, retrain up to 116, test on
  117. Rolling mode repeats this for folds 112-117. Congress 118 is a locked final test, off by default.

**Run:** install `notebooks/requirements.txt`, edit only the config cell, then *Run All*.
About 12 minutes on a laptop CPU in simple mode.

In [ ]:
import os
import sys
from pathlib import Path

import pandas as pd

for _p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:     # make `import baselines` work from any folder
    if (_p / "baselines" / "__init__.py").exists():
        sys.path.insert(0, str(_p)); break
    if (_p / "notebooks" / "baselines" / "__init__.py").exists():
        sys.path.insert(0, str(_p / "notebooks")); break

from baselines import checks, config, data, features, graphs, report, train
from baselines.models import available_models, heuristics

pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 30)

## 0. Configuration
The only cell you need to edit.

In [ ]:
MODE = "simple"                    # "simple" = fold 117 | "rolling" = folds 112-117
TASKS = ("t1", "t2")               # t1 = cosponsor prediction, t2 = bill advancement
MODELS = ("hetero_sage", "rgcn", "lightgcn", "pair_mlp")
SEEDS = (0,)                       # (0, 1, 2) to compare learned models fairly (about 3x slower)
T0_LIST = (0,)                     # cosponsor prediction time(s) in days after introduction; (0, 7) adds day 7
T1_PROTOCOLS = ("stream",)         # add "plain" to also train without current-Congress signals
PLAIN_MODELS = ("hetero_sage",)    # models trained under "plain" when it is listed above
DEVICE, THREADS = "cpu", 8         # lower THREADS if other heavy jobs are running
FOLDS_ONLY = ()                    # rolling mode: a subset of folds, e.g. (116,)
ROLLING_CONFIG = "apriori"         # rolling mode: "apriori" (default settings) | "search" | "frozen"
LOCKED_118 = False                 # score the locked test Congress 118; only once, at the very end
RUN_SEARCH = False                 # re-run the hyperparameter search on validation (slow)
USE_TITLE = True                   # use the introduced-title embedding as a bill feature
QUICK = False                      # 2-epoch smoke test

cfg = config.RunConfig(mode=MODE, tasks=TASKS, t0_list=T0_LIST, t1_protocols=T1_PROTOCOLS, plain_models=PLAIN_MODELS,
                       models=MODELS, seeds=SEEDS, device=DEVICE, threads=THREADS, run_search=RUN_SEARCH,
                       rolling_config=ROLLING_CONFIG, locked_118=LOCKED_118, folds_only=FOLDS_ONLY,
                       use_title=USE_TITLE, quick=QUICK)
cfg = config.apply_env_overrides(cfg)    # used only by the headless test runner
train.set_threads(cfg.threads)
FOLDS = cfg.folds() + cfg.locked_folds()
RESULTS = []                             # every result of this run (validation and test)
timer = report.Timer()
print(f"folds {FOLDS} | tasks {cfg.tasks} | models {cfg.models} | seeds {cfg.seeds}" + (" | QUICK" if cfg.quick else ""))

## 1. Data and labels
Loads the Parquet tables and builds both labels, censoring, point-in-time parties and the eligible pools
(identical to the split notebook; a test checks this). One row per Congress below.

In [ ]:
with timer("data"):
    corpus = data.load_corpus(cfg)
    LABELS = data.label_summary(corpus, sorted(set(range(config.SUP_START - 1, max(FOLDS) + 1))), t0=0)
report.show(LABELS)

## 2. Leak-free graphs
One graph per target Congress: legislators, committees and the bills of the 3 previous Congresses (edges both
ways), plus the target bills, which only receive edges dated on or before their cutoff (sponsor, cosponsors so
far, committee referrals). Then the leak checks: if any check fails, the cell raises an error.

In [ ]:
FOLD_DATA = {}
with timer("graphs + features"):
    for c in FOLDS:
        for t0 in cfg.t0_list:
            for fit in ("val", "refit"):
                FOLD_DATA[(c, t0, fit)] = train.fold_data(corpus, c, fit, t0, cfg)
_c, _t0 = FOLDS[0], cfg.t0_list[0]
_d = FOLD_DATA[(_c, _t0, "refit")]
print(graphs.describe_graph(_d.graphs[_c]))

with timer("checks"):
    for (c, t0, fit), d in FOLD_DATA.items():
        assert set(d.checks) == set(d.congresses)          # graph asserts ran on every graph
        for k in d.congresses:
            checks.check_t2_plain(d.graphs[k], corpus, k, d.H)
    checks.check_eligibility(corpus, data.t1_blocks(corpus, _c, _t0))
    checks.check_allowlist(_d.graphs[_c].feature_names | {"phi": list(features.PHI_ALL)})
    checks.check_depth(range(config.SUP_START, max(FOLDS) + 1), cfg.H)
    checks.check_caucus_pit(corpus, _c, cfg.H)
    checks.check_censored_eb(corpus, _c - 1)
    checks.truncation_test(corpus, _c, _t0, cfg.H, title=_d.title)
print("leak checks passed: graph structure, eligibility, feature allowlist, history depth, truncation test")

## 3. Heuristic baselines
The bars a learned model has to clear, ported from the split study.
- **Cosponsor prediction:** popularity; same party, then popularity; **sponsor history** (how often the
  candidate backed this sponsor before); 2-hop random walk; SVD embeddings; co-support with the day-0 cosponsors.
- **Bill advancement:** random prior; committee report rate (and two simpler rate variants).

In [ ]:
with timer("heuristics"):
    for c in FOLDS:
        for t0 in cfg.t0_list:
            if "t1" in cfg.tasks:
                RESULTS += heuristics.t1_reference_results(corpus, c, t0, protocols=("stream", "plain"), H=cfg.H)
        if "t2" in cfg.tasks:
            RESULTS += heuristics.t2_reference_results(corpus, c)
if 117 in FOLDS and 0 in cfg.t0_list and not cfg.quick:       # sanity check against the split study
    checks.check_reference_bars([r for r in RESULTS if r.kind != "gnn"], strict=True)
    print("heuristic scores match the split study")

## 4. HeteroSAGE
GraphSAGE applied per edge type: each node averages its neighbours separately for each kind of edge
(sponsor, cosponsor, referral, ...) and combines them with its own features. Works from features, not IDs, so it
handles new legislators. 2 layers, hidden size 128.

In [ ]:
G1 = report.run_model_section("hetero_sage", cfg, corpus, timer)
RESULTS += G1

## 5. R-GCN
Each edge type gets its own learned transformation; the 11 edge types share 4 basis matrices to keep the model
small. The encoder family used by prior work on cosponsorship. 2 layers, hidden size 128.

In [ ]:
G2 = report.run_model_section("rgcn", cfg, corpus, timer)
RESULTS += G2

## 6. LightGCN-feat
No learning inside the message passing: node features are spread over the graph with a fixed normalized
average, and an MLP learns from the result. Tests whether learned message passing is needed at all.

In [ ]:
G3 = report.run_model_section("lightgcn", cfg, corpus, timer)
RESULTS += G3

## 7. Learned references without a graph
- **PairMLP:** the same model with message passing turned off. The gap to the GNNs is what the graph adds.
- **Gradient-boosted trees / logistic regression** (advancement only): tabular models on the bill's own features.

In [ ]:
PAIR = report.run_model_section("pair_mlp", cfg, corpus, timer)
RESULTS += PAIR
TAB = []
if "t2" in cfg.tasks and cfg.tabular_refs:
    with timer("tabular models"):
        for c in FOLDS:
            for seed in cfg.seeds:
                TAB += heuristics.t2_tabular_results(corpus, c, cfg.tab_sup_start, cfg.H, cfg.use_title, seed)
    RESULTS += TAB

## 8. Results
One table per task: validation score (Congress c-1) and test score (Congress c) with a 95% bootstrap CI.
With one seed, differences between learned models below about 0.015 are noise; the cached 3-seed results are
printed at the end for comparison.

In [ ]:
DF = report.results_frame(RESULTS, corpus)
for c in FOLDS:
    for t0 in cfg.t0_list:
        if "t1" in cfg.tasks:
            report.show(report.comparison_table(DF, "t1", c, t0), f"\nCosponsor prediction, fold {c}, t0 = {t0}: Recall@20")
    if "t2" in cfg.tasks:
        report.show(report.comparison_table(DF, "t2", c), f"\nBill advancement, fold {c}: AUROC")

In [ ]:
# Paired test differences (real = difference >= 0.01 and every seed's CI excludes 0), and new legislators.
checks.check_population(RESULTS)
for _w in checks.direction_warnings(RESULTS, cfg.plain_models):
    print("WARNING", _w)
PAIRED = report.show_paired(RESULTS, corpus)
if len(PAIRED):
    _ps = report.paired_seed_summary(PAIRED)
    report.show(_ps[["task", "model", "vs", "metric", "diff_mean", "real"]], "\nPaired differences (test):")
NEWLEG = report.new_legislator_table(RESULTS)
if len(NEWLEG):
    _nl = NEWLEG[NEWLEG.fold.isin(cfg.folds()[-1:]) & (NEWLEG.protocol == "stream")]
    report.show(_nl[["model", "new_hit20", "returning_hit20"]].sort_values("new_hit20"),
                "\nHit@20 for new vs returning legislators (cosponsor prediction):")
print(f"\ntotal runtime: {timer.table().loc['total (sum of stages)', 'minutes']:.1f} min")

In [ ]:
OUT_DIR = Path(os.environ.get("BASELINES_OUT", corpus.root / "notebooks" / "baselines_results" / cfg.mode))
report.save_results(DF, OUT_DIR, timer, extra=dict(config=config.config_dict(cfg), folds=FOLDS))
if len(PAIRED):
    PAIRED.to_csv(OUT_DIR / "paired.csv", index=False)
_ref = corpus.root / "notebooks" / "baselines_results" / "reference_fold117_3seeds"
if (_ref / "results.csv").exists() and len(cfg.seeds) < 2:
    _rdf = pd.read_csv(_ref / "results.csv")
    for _task, _name in (("t1", "Cosponsor prediction"), ("t2", "Bill advancement")):
        if _task in cfg.tasks:
            report.show(report.comparison_table(_rdf, _task, 117, 0), f"\nCached 3-seed results, {_name}, fold 117:")

## 9. Rolling folds and the locked test
Only runs with `MODE = "rolling"` or `LOCKED_118 = True`. Rolling mode prints each test fold and the mean over
folds (paired t-test against the best heuristic). Congress 118 is printed on its own.

In [ ]:
if cfg.mode == "rolling":
    _dev = DF[DF.fold.isin(cfg.folds())]
    for _task, _base, _m in (("t1", "sponsor_hist", "Recall@20"), ("t2", "eb_rate_t0safe", "AUROC")):
        if _task in cfg.tasks:
            _per = _dev[(_dev.task == _task) & (_dev.role == "test") & (_dev.metric == _m)].pivot_table(
                index=["t0", "protocol", "model", "config_set", "selection"], columns="fold", values="value")
            report.show(_per, f"\n{_task} {_m} per test fold:")
            _rs = report.rolling_summary(_dev[_dev.task == _task], baseline=_base)
            report.show(_rs[_rs.metric == _m], f"\n{_task}: mean over folds vs {_base} (folds tuned on test data excluded):")
if cfg.locked_118:
    _l = DF[(DF.fold == 118) & (DF.role == "test")]
    for _task in cfg.tasks:
        report.show(report.show_main(_l, _task, "test", fold=118), f"\nLocked test Congress 118, {_task}:")
if cfg.mode != "rolling" and not cfg.locked_118:
    print("simple mode: nothing to do here")

## 10. What to experiment with
Change one thing at a time in `baselines/config.py` and judge it on the **validation** score only.

| Knob | Default | Try |
|---|---|---|
| `SEEDS` | (0,) | (0, 1, 2) before ranking learned models |
| dropout (HeteroSAGE, advancement) | 0.2 | 0.3-0.5 with weight decay 1e-3 |
| `RGCNConfig.num_bases` | 4 | 2-8 |
| history window `RunConfig.H` | 3 | 2-4 |
| layers | 2 | 0-3 (0 = PairMLP) |
| `TrainConfig.lr` | 3e-3 (1e-3 LightGCN) | 1e-3 to 6e-3 |
| cosponsor loss `TrainConfig.loss` | "softmax" | "bpr" |
| `T0_LIST` / `T1_PROTOCOLS` | (0,) / ("stream",) | (0, 7) / ("stream", "plain") |

Set `RUN_SEARCH = True` to re-run the validation search. Never choose settings from test scores.

In [ ]:
if cfg.run_search:
    for _m in [m for m in cfg.models if m in available_models()]:
        for _task in cfg.tasks:
            _log = train.run_search(_m, _task, cfg.folds()[-1], cfg, corpus)
            print(f"{_m} {_task}: chosen {_log['chosen']} (val {_log['best_val']:.4f})")